#### Simple Chatbot using Langchain

In [1]:
import os
from dotenv import load_dotenv

# This single line loads all variables from .env into os.environ automatically
load_dotenv()

# Hardcode this one manually ONLY if you didn't put it in your .env file
os.environ["LANGCHAIN_TRACING_V2"] = "false"

In [2]:
## Data Ingestion--From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader

C:\Users\manav\AppData\Local\Temp\ipykernel_5104\2433114369.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
loader=WebBaseLoader("https://en.wikipedia.org/wiki/Indian_Premier_League")
loader

In [4]:
docs=loader.load()
docs

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Indian_Premier_League', 'title': 'Indian Premier League - Wikipedia', 'language': 'en'}, page_content='\n\n\n\nIndian Premier League - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\n\nDonate\n\n\nCreate account\n\n\nLog in\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nContents\nmove to sidebar\nhide\n\n\n\n\n(Top)\n\n\n\n\n\n1\nHistory\n\n\n\n\nToggle History sub

In [5]:
### Load Data--> Docs-->Divide our Docuemnts into chunks dcouments-->text-->vectors-->Vector Embeddings--->Vector Store DB
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)

In [6]:
documents

[Document(metadata={'source': 'https://en.wikipedia.org/wiki/Indian_Premier_League', 'title': 'Indian Premier League - Wikipedia', 'language': 'en'}, page_content='Indian Premier League - Wikipedia\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nJump to content\n\n\n\n\n\n\n\nMain menu\n\n\n\n\n\nMain menu\nmove to sidebar\nhide\n\n\n\n\t\tNavigation\n\t\n\n\nMain pageContentsCurrent eventsRandom articleAbout WikipediaContact us\n\n\n\n\n\n\t\tContribute\n\t\n\n\nHelpLearn to editCommunity portalRecent changesUpload fileSpecial pages\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\nSearch\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nAppearance\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nDonate\n\nCreate account\n\nLog in\n\n\n\n\n\n\n\n\nPersonal tools\n\n\n\n\n\n\nDonate\n\n\nCreate account\n\n\nLog in\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nContents\nmove to sidebar\nhide\n\n\n\n\n(Top)\n\n\n\n\n\n1\nHistory\n\n\n\n\nToggle History subsection\

In [7]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()


embeddings = OpenAIEmbeddings(
    base_url=f"{os.environ.get('OPENROUTER_BASE_URL')}/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    model="openai/text-embedding-3-small" 
)

In [8]:
from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

In [10]:
vectorstoredb

In [11]:
## Query From a vector db
query="When IPL introduced the strategic timeout"
result=vectorstoredb.similarity_search(query)
result[0].page_content

'Strategic timeouts\nIn the 2009 season, the IPL introduced the "strategic timeout", a seven-and-a-half minute break at the end of the tenth over in each innings.[181] Teams and players, including Sachin Tendulkar, disapproved of it for interrupting the flow of play,[182] while critics labelled it a "commercial use of an extended drinks break" as every 10-second advertisement slot was sold for over ₹5 lakh.[183][184] The then-president of the IPL insisted that the break was intended to allow teams to strategise during the game. Following widespread criticism, in the 2010 season, the duration of the timeout was reduced to five minutes, split in two halves during each innings.[184][185]'

In [12]:
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    base_url=f"{os.environ.get('OPENROUTER_BASE_URL')}/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    model="openrouter/free", # Hardcode the free auto-router here!
    max_tokens=1000  
)

In [14]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document


# 1. Your existing setup
prompt = ChatPromptTemplate.from_template(
    """
    Answer the following question based only on the provided context:
    <context>
    {context}
    </context>
    
    Question: {input}
    """
)

document_chain = create_stuff_documents_chain(llm, prompt)

In [15]:
import langsmith as ls
from langchain_core.documents import Document

# Disables tracing for just this specific execution block
with ls.tracing_context(enabled=False):
    response = document_chain.invoke({
        "input": "LangSmith has two usage limits: total traces and extended",
        "context": [Document(page_content="LangSmith has two usage limits: total traces and extended traces. These correspond to the two metrics we've been tracking on our usage graph. ")]
    })

print(response)


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


The two usage limits are **total traces** and **extended traces**. They correspond to the two metrics tracked on the usage graph.


In [16]:
### Input--->Retriever--->vectorstoredb

vectorstoredb


In [17]:
retriever = vectorstoredb.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [18]:
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002582EB497F0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context:\n    <context>\n    {context}\n    </context>\n\n    Questi

In [19]:
## Get the response form the LLM
response=retrieval_chain.invoke({"input":"When IPL introduced the strategic timeout"})
response['answer']

'The IPL introduced the strategic timeout in the 2009 season.'

In [20]:
response

{'input': 'When IPL introduced the strategic timeout',
 'context': [Document(id='665de09d-deae-42c6-9dbc-8f4ab75b521e', metadata={'source': 'https://en.wikipedia.org/wiki/Indian_Premier_League', 'title': 'Indian Premier League - Wikipedia', 'language': 'en'}, page_content='Strategic timeouts\nIn the 2009 season, the IPL introduced the "strategic timeout", a seven-and-a-half minute break at the end of the tenth over in each innings.[181] Teams and players, including Sachin Tendulkar, disapproved of it for interrupting the flow of play,[182] while critics labelled it a "commercial use of an extended drinks break" as every 10-second advertisement slot was sold for over ₹5 lakh.[183][184] The then-president of the IPL insisted that the break was intended to allow teams to strategise during the game. Following widespread criticism, in the 2010 season, the duration of the timeout was reduced to five minutes, split in two halves during each innings.[184][185]'),
  Document(id='ed4ecbfc-e051-4

In [21]:
response['context']


[Document(id='665de09d-deae-42c6-9dbc-8f4ab75b521e', metadata={'source': 'https://en.wikipedia.org/wiki/Indian_Premier_League', 'title': 'Indian Premier League - Wikipedia', 'language': 'en'}, page_content='Strategic timeouts\nIn the 2009 season, the IPL introduced the "strategic timeout", a seven-and-a-half minute break at the end of the tenth over in each innings.[181] Teams and players, including Sachin Tendulkar, disapproved of it for interrupting the flow of play,[182] while critics labelled it a "commercial use of an extended drinks break" as every 10-second advertisement slot was sold for over ₹5 lakh.[183][184] The then-president of the IPL insisted that the break was intended to allow teams to strategise during the game. Following widespread criticism, in the 2010 season, the duration of the timeout was reduced to five minutes, split in two halves during each innings.[184][185]'),
 Document(id='ed4ecbfc-e051-4286-aaca-fc48076f2c76', metadata={'source': 'https://en.wikipedia.or

In [22]:
## Get the response form the LLM
response=retrieval_chain.invoke({"input":"how many teams are there in IPL"})
response['answer']

'The Indian Premier League (IPL) currently consists of **10 teams**. This was confirmed by the expansion in 2022 with the addition of Gujarat Titans and Lucknow Super Giants, and the context explicitly states, "As of the 2025 season, the league consists of 10 teams."'